# Train and evaluate the chest X-ray classifier on Colab

**For education only, not for clinical use.**

DenseNet121 (ImageNet weights), 14 NIH findings, multi-label BCE, fixed config
`configs/classifier.yaml`: batch 64, AdamW 1e-4, at most 5 epochs, AMP.

- **Not run yet.** The code is tested on tiny synthetic data on CPU only.
- Use a **GPU runtime** (T4 is enough).
- Needs the `preprocess` outputs in the DVC remote on Drive: run `colab_preprocess.ipynb` first
  and commit its `dvc.lock` (README, "Data versioning").
- **Storage:** `dvc pull` copies the images to the VM disk, and training reads only from there.
  Drive holds the remote, the checkpoints and the reports.
- **Resumable:** a checkpoint per epoch goes to `MyDrive/aiml02-runs/classifier/` and is never
  overwritten. After a disconnect, run all cells again: training continues after the latest
  checkpoint.
- The best epoch is the one with the highest val mean AUROC. Thresholds are chosen on val only.
- Results go to `MyDrive/aiml02-runs/classifier/reports/`; copy them into the repo afterwards.

## 1. GPU, Drive and disk

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path


def run(*args, cwd=None, env=None):
    """Run a command, stream its output into the notebook, fail on a non-zero exit."""
    process = subprocess.Popen(
        [str(arg) for arg in args], cwd=cwd, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True,
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(f"{args[0]} failed with exit code {process.returncode}")


def free_gb(path):
    return shutil.disk_usage(path).free / 1e9


import torch

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

from google.colab import drive

drive.mount("/content/drive")
run("df", "-h", "/content", "/content/drive")
NEED_VM_GB = 12  # shards (the DVC cache hardlinks them), with margin
print(f"VM: {free_gb('/content'):.1f} GB free, {NEED_VM_GB} GB needed")
if free_gb("/content") < NEED_VM_GB:
    raise RuntimeError("Not enough VM disk. Stop here.")
NEED_DRIVE_GB = 1  # five checkpoints of about 100 MB
if free_gb("/content/drive") < NEED_DRIVE_GB:
    raise RuntimeError("Not enough Drive space for the checkpoints. Stop here.")

## 2. Code from the repo, and versions

In [ ]:
REPO_URL = "https://github.com/manojkumargudge/medical-imaging-ai.git"
REPO_REF = "main"
REPO = Path("/content/medical-imaging-ai")

if not REPO.exists():
    run("git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, REPO)
commit = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
print("repo commit", commit)
# On the path rather than installed: the repo pins are for local Python 3.11, and Colab's own
# packages (Pillow, PyYAML, pandas, torch with CUDA) are used here.
sys.path.insert(0, str(REPO))
os.chdir(REPO)

run(sys.executable, "-m", "pip", "install", "-q", "dvc==3.67.1")
remote = subprocess.run(
    ["dvc", "remote", "list"], cwd=REPO, capture_output=True, text=True, check=True
).stdout
print(remote)
assert "/content/drive/MyDrive/aiml02-dvc" in remote, "DVC remote is not the Drive folder"
# The cache hardlinks to the workspace instead of holding a second copy on the VM disk.
run("dvc", "config", "--local", "cache.type", "hardlink,copy", cwd=REPO)

import re

import torchvision

pins = dict(re.findall(r'"(torch|torchvision)==([^"]+)"', (REPO / "pyproject.toml").read_text()))
for name, version in (("torch", torch.__version__), ("torchvision", torchvision.__version__)):
    note = "" if version.split("+")[0] == pins[name] else f" (repo pin {pins[name]}; recorded)"
    print(f"{name} {version}{note}")

## 3. `dvc pull` from the Drive remote to the VM disk

In [ ]:
lock = (REPO / "dvc.lock").read_text()
if "preprocess:" not in lock:
    raise RuntimeError("dvc.lock has no preprocess stage: commit it from the preprocess run first")
run("dvc", "pull", "data/processed/nih_224", "data/manifests/nih_224.csv", cwd=REPO)
run("df", "-h", "/content")

from src.data import preprocess
from src.models.config import load_config
from src.models.data import read_manifest

config = load_config(REPO / "configs" / "classifier.yaml")
# Train only from the VM disk: Drive is too slow and holds only the remote and the checkpoints.
for path in (REPO / config.processed_dir, REPO / config.manifest):
    assert not str(path.resolve()).startswith("/content/drive"), f"{path} is on Drive"
verified = preprocess.verify_shards(REPO / config.processed_dir)
samples = read_manifest(REPO / config.manifest, config.labels)  # prints counts, checks patients
rows = sum(len(items) for items in samples.values())
print(f"shard images {verified.images}, manifest rows {rows}")
assert verified.images > 0 and rows > 0 and verified.images == rows

## 4. Train (resumes after the latest checkpoint)

About 78k train images per epoch. Each epoch ends with a val pass, a checkpoint and a small JSON
file with the epoch's metrics.

In [ ]:
from src.models import train

print("checkpoints in", config.checkpoint_dir)
result = train.train(config)
print("best epoch", result["best"])

## 5. Evaluate the best epoch on val and test

Per-label AUROC with positive counts, thresholds chosen on val (highest F1) and applied to test,
and AUROC by sex and age band (ages above 100 are counted as invalid and not scored). An existing
report is not overwritten.

In [ ]:
from src.models import evaluate

report = evaluate.evaluate(config, meta={"repo_commit": commit})
REPORTS = config.checkpoint_dir / "reports"
evaluate.write_outputs(
    report,
    REPORTS / "classifier_metrics.json",
    REPORTS / "classifier_thresholds.json",
    REPORTS / "results.md",
)
print((REPORTS / "results.md").read_text())

## 6. Afterwards, on the laptop

With Google Drive for desktop (README, "Data versioning"), copy from
`My Drive/aiml02-runs/classifier/reports/`:

- `classifier_metrics.json` and `classifier_thresholds.json` to `reports/`
- `results.md` to `docs/results.md`

then commit them. The checkpoints stay on Drive.